# 02 - Model comparison (2 baselines + 4 deep), K-fold CV

Compares every model trained against the 50 km-grid cache (`cache/goes_grid50_2019_2026`)
under **blocked K-fold cross-validation** (see `foldsplit.py`): a single **fixed test set**
(interleaved months, spans all of 2019-2025) plus **6 CV folds** over the rest, each fold
rotating one block as **val** (~15%) and training on the others (~75%). Every fold's train,
val, and the fixed test span all 7 years with matched base rates.

- **baselines** (per-cell 168-feature vectors): logistic regression, XGBoost
- **deep** (CNN / attention over the 59x95 feature grid): 3D-ResNet, 3D-CNN,
  CNN+temporal-attention, ConvGRU (+attn)

Each fold writes `outputs/<name>_f<k>.pt`/`.pkl` (+ deep `<name>_f<k>_results.npz`). This
notebook reloads all folds, predicts on each fold's val and on the fixed test, and reports
**AUPRC mean ± std across folds** (val for model selection, fixed test for the honest
comparison), a **6-fold ensemble** on the test set, training curves, and prediction maps.


## 0. Setup, registry, and artifact availability

In [ ]:
import os
import pickle
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve
from torch.utils.data import DataLoader

ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
MODEL_DIR = ROOT / "notebooks" / "model"
OUT_DIR = MODEL_DIR / "outputs"
sys.path.insert(0, str(MODEL_DIR))
sys.path.insert(0, str(MODEL_DIR / "trainers"))

from config import STATES_GEOJSON, build_grid_cells
from gridindex import build_pix2cell
import foldsplit
import resnet3d, cnn_attn, convgru_attn   # noqa: E401  (deep)
import xgb                                 # noqa: E401  (tabular)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
N_FOLDS = foldsplit.N_FOLDS

p2c, GRID_R, GRID_C, land = build_pix2cell()
CACHE_DIR = resnet3d.CACHE_DIR


def fold_days(k):
    """Set the fold and return (train, val, test); test is fixed across folds."""
    os.environ["FOLD"] = str(k)
    return foldsplit.fold_splits(CACHE_DIR)


_, _, te_days = fold_days(0)                    # fixed test set (same every fold)


def base_rate(days):
    y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in days])
    return float(y[:, land].mean())


BASE_TEST = base_rate(te_days)

# grid polygons + state outlines (Albers) for the maps
cells_gdf, _, _, _ = build_grid_cells()
cells_albers = cells_gdf.to_crs(5070)
RR, CC = cells_albers["R"].to_numpy(), cells_albers["C"].to_numpy()
states = gpd.read_file(STATES_GEOJSON)
states = states[~states["name"].isin(["Alaska", "Hawaii", "Puerto Rico"])].to_crs(5070)

# 7-model registry: deep (reload .pt into FloodNet) + tabular (reload .pkl), per fold
MODELS = {
    "3D-ResNet":    dict(kind="deep", mod=resnet3d,     stem="resnet3d",     c="#d62828", cmap="Reds"),
    "CNN+attn":     dict(kind="deep", mod=cnn_attn,     stem="cnn_attn",     c="#2a9d8f", cmap="GnBu"),
    "ConvGRU+attn": dict(kind="deep", mod=convgru_attn, stem="convgru_attn", c="#118ab2", cmap="PuBu"),
    "XGBoost":      dict(kind="tab",  mod=xgb,          stem="xgb",          c="#386641", cmap="YlGn"),
}


def fold_ckpt(mi, k):
    ext = "pt" if mi["kind"] == "deep" else "pkl"
    return OUT_DIR / f"{mi['stem']}_f{k}.{ext}"


def folds_avail(mi):
    return [k for k in range(N_FOLDS) if fold_ckpt(mi, k).exists()]


AVAIL = [n for n, mi in MODELS.items() if folds_avail(mi)]

print(f"device {DEVICE} | grid {GRID_R}x{GRID_C} | land {int(land.sum())} cells | "
      f"{N_FOLDS}-fold CV")
print(f"fixed test: {len(te_days)} days  base rate {BASE_TEST:.4f}")
for n, mi in MODELS.items():
    fa = folds_avail(mi)
    print(f"  {n:12s} {mi['kind']:4s} -> folds {fa if fa else 'none (train it)'}")


## 1. Training curves (deep models)

Train loss and **train/val** PR-AUC across epochs from each `*_results.npz` (early-stopped on
val). Test is intentionally **not** tracked per epoch (only at the end, in the metrics table)
to avoid biasing model selection. Dotted line = val base rate.

In [ ]:
deep_avail = [n for n in AVAIL if MODELS[n]["kind"] == "deep"]

for name in deep_avail:
    stem = MODELS[name]["stem"]
    fig, (a_loss, a_pr) = plt.subplots(1, 2, figsize=(13, 4.4))
    bests = []
    for k in range(N_FOLDS):
        path = OUT_DIR / f"{stem}_f{k}_results.npz"
        if not path.exists():
            continue
        h = np.load(path)["hist"]              # epoch, lr, loss, tr_pr, va_pr
        if h.ndim != 2 or h.shape[1] != 5 or len(h) == 0:
            continue
        ep = h[:, 0]
        a_loss.plot(ep, h[:, 2], "-", lw=1, alpha=0.7, label=f"fold {k}")
        a_pr.plot(ep, h[:, 4], "-", lw=1, alpha=0.85, label=f"fold {k}")
        bests.append(float(h[:, 4].max()))
    a_pr.axhline(BASE_TEST, ls=":", color="black", lw=1.2,
                 label=f"test base ({BASE_TEST:.4f})")
    a_loss.set(xlabel="epoch", ylabel="combined loss",
               title=f"{name} - train loss (per fold)")
    a_pr.set(xlabel="epoch", ylabel="val PR-AUC",
             title=f"{name} - val PR-AUC (per fold)")
    for ax in (a_loss, a_pr):
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    if bests:
        print(f"{name}: best val PR-AUC {np.mean(bests):.4f} ± {np.std(bests):.4f} "
              f"across {len(bests)} folds")


## 2. Inference on val + test

Reload each trained model and predict per-cell probabilities `(N, 59, 95)`. Deep models
reload their `.pt` into `FloodNet`; tabular models reload their `.pkl` and use
`predict_grids`. Cached once for the metrics table and maps below.

In [ ]:
@torch.no_grad()
def infer(mi, ckpt, days):
    """Reload one fold's model and predict -> (probs, trues), each (len(days),59,95)."""
    if mi["kind"] == "deep":
        mod = mi["mod"]
        net = mod.FloodNet().to(DEVICE)                    # stats are saved buffers
        net.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        net.eval()
        probs, trues = [], []
        loader = DataLoader(mod.FeatureCache(days), batch_size=16, num_workers=8)
        for seq, summ, t, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                p = torch.sigmoid(net(seq.to(DEVICE).float(), summ.to(DEVICE).float(),
                                      t.to(DEVICE).float())).squeeze(1)
            probs.append(p.float().cpu().numpy())
            trues.append(y.numpy())
        del net
        torch.cuda.empty_cache()
        return np.concatenate(probs), np.concatenate(trues)
    payload = pickle.load(open(ckpt, "rb"))                 # tabular
    return mi["mod"].predict_grids(payload, days, land)


# per-fold predictions: each fold's own val + the fixed test; plus a 6-fold ensemble
fold_val, fold_test, ens_test = {}, {}, {}
for name in AVAIL:
    mi = MODELS[name]
    fold_val[name], fold_test[name] = [], []
    print(f"inferring {name} folds:", end=" ", flush=True)
    t_probs = []
    for k in folds_avail(mi):
        _, va_k, _ = fold_days(k)                           # fold-k val
        fold_val[name].append(infer(mi, fold_ckpt(mi, k), va_k))
        tp = infer(mi, fold_ckpt(mi, k), te_days)           # fixed test
        fold_test[name].append(tp)
        t_probs.append(tp[0])
        print(k, end=" ", flush=True)
    ens_test[name] = (np.mean(t_probs, axis=0), fold_test[name][0][1])
    print("done")


## 3. Metrics table - K-fold CV

For each model, every fold's threshold is the best-F1 point on **that fold's val**, applied
to the **fixed test set**. We report **AUPRC as mean ± std across the folds** (val =
model-selection metric, test = honest comparison), plus the **6-fold ensemble** (mean of the
fold-models' test probabilities) as the single best achievable predictor. `xbase` = AUPRC /
fixed-test base rate.


In [ ]:
import pandas as pd


def _dilate_1grid(mask):
    out = mask.copy()
    out[:-1, :] |= mask[1:, :]
    out[1:, :] |= mask[:-1, :]
    out[:, :-1] |= mask[:, 1:]
    out[:, 1:] |= mask[:, :-1]
    out[:-1, :-1] |= mask[1:, 1:]
    out[1:, 1:] |= mask[:-1, :-1]
    out[:-1, 1:] |= mask[1:, :-1]
    out[1:, :-1] |= mask[:-1, 1:]
    return out


def full_metrics(probs, trues, threshold=None):
    p = probs[:, land].ravel()
    t = trues[:, land].ravel().astype(np.int32)
    prauc = average_precision_score(t, p)
    if threshold is None:
        pr_c, rc_c, thr_c = precision_recall_curve(t, p)
        f1_c = 2 * pr_c * rc_c / (pr_c + rc_c + 1e-9)
        threshold = float(thr_c[np.argmax(f1_c[:-1])])
    yb = (p >= threshold).astype(np.int32)
    tp = int((yb * t).sum()); fp = int((yb * (1 - t)).sum()); fn = int(((1 - yb) * t).sum())
    prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9); csi = tp / (tp + fn + fp + 1e-9)
    return dict(prauc=prauc, thr=threshold, prec=prec, rec=rec, f1=f1, csi=csi)


# aggregate across folds; store ensemble prediction + mean threshold for the maps
rows, THR, test_pred = [], {}, {}
for name in AVAIL:
    va, ta, thrs, tf1, tcsi = [], [], [], [], []
    for vp, tp in zip(fold_val[name], fold_test[name]):
        vm = full_metrics(*vp, threshold=None)              # fold val -> best-F1 thr
        tm = full_metrics(*tp, threshold=vm["thr"])         # fixed test @ that thr
        va.append(vm["prauc"]); ta.append(tm["prauc"]); thrs.append(vm["thr"])
        tf1.append(tm["f1"]); tcsi.append(tm["csi"])
    THR[name] = float(np.mean(thrs))
    test_pred[name] = ens_test[name]                        # (mean probs, trues)
    em = full_metrics(*ens_test[name], threshold=THR[name])  # 6-fold ensemble
    rows.append(dict(model=name, folds=len(va),
                     val_AUPRC=np.mean(va), val_sd=np.std(va),
                     test_AUPRC=np.mean(ta), test_sd=np.std(ta),
                     test_xbase=np.mean(ta) / BASE_TEST,
                     test_F1=np.mean(tf1), test_CSI=np.mean(tcsi),
                     ens_AUPRC=em["prauc"], ens_xbase=em["prauc"] / BASE_TEST,
                     ens_F1=em["f1"], ens_CSI=em["csi"]))

tbl = pd.DataFrame(rows).sort_values("test_AUPRC", ascending=False)
test_tbl = tbl                                              # alias for maps below

disp = tbl.copy()
disp["val AUPRC"] = disp.apply(lambda r: f"{r.val_AUPRC:.4f} ± {r.val_sd:.4f}", axis=1)
disp["test AUPRC"] = disp.apply(
    lambda r: f"{r.test_AUPRC:.4f} ± {r.test_sd:.4f} ({r.test_xbase:.1f}x)", axis=1)
disp["ens test AUPRC"] = disp.apply(
    lambda r: f"{r.ens_AUPRC:.4f} ({r.ens_xbase:.1f}x)", axis=1)
cols = ["folds", "val AUPRC", "test AUPRC", "ens test AUPRC",
        "test_F1", "test_CSI", "ens_F1", "ens_CSI"]
print(f"fixed-test base rate {BASE_TEST:.4f}  |  {N_FOLDS}-fold CV (mean ± sd)\n")
display(disp.set_index("model")[cols].round(4))
_b = tbl.iloc[0]
print(f"\n>>> best mean test AUPRC: {_b['model']}  "
      f"{_b['test_AUPRC']:.4f} ± {_b['test_sd']:.4f}  ({_b['test_xbase']:.1f}x base)")
tbl.round(4)


## 4. Prediction maps — random test days

Grid of **rows = models** (ground truth on top) × **columns = sample test days**. Change
`SEED` to resample which days are shown; `N_SHOW` sets how many. This cell binarizes each
model's prediction at its best-val-F1 threshold; the next cell shows the raw probability
heatmap. Per-panel AUPRC is annotated below each map.

In [ ]:
assert AVAIL, "no trained models yet - train them first (see section 0)"
SEED = 99        # <- change to resample which test days are shown
N_SHOW = 5


def _draw(ax, values, cmap, norm=None, title=None, ylabel=None, sub=None):
    gdf = cells_albers.copy(); gdf["v"] = values[RR, CC]
    gdf.boundary.plot(ax=ax, color="white", lw=0.1, zorder=2)
    kw = dict(norm=norm) if norm is not None else dict(vmin=0, vmax=1)
    gdf.plot(column="v", cmap=cmap, ax=ax, zorder=1, edgecolor="none", **kw)
    states.boundary.plot(ax=ax, color="0.5", lw=0.5, zorder=3)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if title:  ax.set_title(title, fontsize=11, fontweight="bold")
    if ylabel: ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")
    if sub:    ax.set_xlabel(sub, fontsize=8, color="0.3")


rng = np.random.default_rng(SEED)
sel = sorted(rng.choice(len(te_days), size=min(N_SHOW, len(te_days)), replace=False))
row_names = ["Ground truth"] + AVAIL                      # rows = GT + each model
nrow, ncol = len(row_names), len(sel)

fig, axes = plt.subplots(nrow, ncol, figsize=(3.5 * ncol, 3.2 * nrow), squeeze=False)
for c, i in enumerate(sel):
    gt = test_pred[AVAIL[0]][1][i]
    yt = gt[land].ravel().astype(int)
    _draw(axes[0, c], gt, "Greens",
          title=f"{te_days[i]}  ({int((gt[land] > 0).sum())} floods)",
          ylabel="Ground truth" if c == 0 else None)
    for r, name in enumerate(AVAIL, start=1):
        pr = test_pred[name][0][i]
        ap = average_precision_score(yt, pr[land].ravel()) if yt.sum() else float("nan")
        _draw(axes[r, c], (pr >= THR[name]).astype(float), MODELS[name]["cmap"],
              ylabel=name if c == 0 else None, sub=f"AUPRC {ap:.3f}")

fig.suptitle(f"Test days (seed {SEED}) - binary predictions @ best-val-F1 threshold",
             fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout(); plt.show()

## 5. Probability heatmaps — same days

Same rows-models × columns-days grid, but each model panel shows the **raw predicted
probability** (not thresholded), on a shared 0-to-max colour scale so models are
comparable. Reuses the `SEED`/`sel` chosen above.

In [ ]:
PCMAP = "viridis"
vmax = max(float(test_pred[n][0][sel][:, land].max()) for n in AVAIL)
vmax = max(vmax, 1e-3)
pnorm = mcolors.Normalize(0, vmax)

fig, axes = plt.subplots(nrow, ncol, figsize=(3.5 * ncol, 3.2 * nrow), squeeze=False)
for c, i in enumerate(sel):
    gt = test_pred[AVAIL[0]][1][i]
    yt = gt[land].ravel().astype(int)
    _draw(axes[0, c], gt, "Greens",
          title=f"{te_days[i]}  ({int((gt[land] > 0).sum())} floods)",
          ylabel="Ground truth" if c == 0 else None)
    for r, name in enumerate(AVAIL, start=1):
        pr = test_pred[name][0][i]
        ap = average_precision_score(yt, pr[land].ravel()) if yt.sum() else float("nan")
        _draw(axes[r, c], pr, PCMAP, norm=pnorm,
              ylabel=name if c == 0 else None, sub=f"AUPRC {ap:.3f}")

fig.subplots_adjust(right=0.9)
cax = fig.add_axes([0.92, 0.15, 0.014, 0.7])
fig.colorbar(plt.cm.ScalarMappable(cmap=PCMAP, norm=pnorm), cax=cax,
             label="predicted probability")
fig.suptitle(f"Test days (seed {SEED}) - predicted probability (shared scale 0-{vmax:.2f})",
             fontsize=14, fontweight="bold", y=1.002)
plt.show()

## 6. Spatial skill of the best model

For the top model (by test AUPRC), aggregate over all test days: **observed**
flood-days per cell vs **predicted** flood-days (binarized at its val threshold),
and the **net bias** (pred - obs). Shows *where* across CONUS the model
over- or under-predicts floods.

In [ ]:
name = test_tbl.iloc[0]["model"]                 # best by test AUPRC
probs, trues = test_pred[name]
obs = trues.sum(0).astype(float)                 # observed flood-days per cell
pred = (probs >= THR[name]).sum(0).astype(float)  # predicted flood-days per cell
bias = pred - obs
mx = max(obs.max(), pred.max(), 1.0)
b = max(abs(bias).max(), 1.0)
cnt_norm = mcolors.Normalize(0, mx)
div_norm = mcolors.TwoSlopeNorm(0, -b, b)

fig, axes = plt.subplots(1, 3, figsize=(4.2 * 3, 4.2))
_draw(axes[0], obs, "magma_r", norm=cnt_norm, title="Observed flood-days")
_draw(axes[1], pred, "magma_r", norm=cnt_norm, title=f"{name}: predicted flood-days")
_draw(axes[2], bias, "RdBu_r", norm=div_norm, title="Net bias (pred - obs)")
fig.colorbar(plt.cm.ScalarMappable(cmap="magma_r", norm=cnt_norm), ax=axes[:2],
             fraction=0.024, pad=0.01, label="flood-days over test")
fig.colorbar(plt.cm.ScalarMappable(cmap="RdBu_r", norm=div_norm), ax=axes[2],
             fraction=0.046, pad=0.01, label="pred - obs")
fig.suptitle(f"Spatial skill of best model ({name}) over {len(te_days)} test days",
             fontsize=13, fontweight="bold")
plt.show()